In [0]:
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, DoubleType, TimestampType, FloatType, DecimalType, IntegerType,DateType
import pyspark.sql.functions as F

catalog_name = 'yamaha-catalog'

In [0]:
df_silver_supersession = spark.table(f'`{catalog_name}`.silver.slv_supersession')
df_silver_supersession.show(10)

+---------------+---------------+--------------------+-------------+--------------+-----+-------------+-------------+-------------+-------------+-------------+-------------+-------------+-------------+-------------+--------------+--------------------+--------------------+
|       Material|      Latest_SS|Material_description|Material_Type|Material_Group|Brand|1st_Supersede|2nd_Supersede|3rd_Supersede|4th_Supersede|5th_Supersede|6th_Supersede|7th_Supersede|8th_Supersede|9th_Supersede|10th_Supersede|        _source_file|         ingested_at|
+---------------+---------------+--------------------+-------------+--------------+-----+-------------+-------------+-------------+-------------+-------------+-------------+-------------+-------------+-------------+--------------+--------------------+--------------------+
|  0NH9-28471-00|  0NH9-28471-00|   BRKT.PATROL LIGHT|         ERSA|     AWPYM0027|   YM|         NULL|         NULL|         NULL|         NULL|         NULL|         NULL|        

### Unt Sales

In [0]:
df_silver_unit_sales = spark.table(f'`{catalog_name}`.silver.slv_unit_sales')
df_silver_unit_sales.show(10)

# Gold unit sales: key business columns in the agreed order (Dealer_code renamed to Dealer_Code)
# The source already carries Year, Month, dealer geography, and customer age - no join or derivation needed
df_gold_unit_sales = (
    df_silver_unit_sales
    .withColumnRenamed('Dealer_code', 'Dealer_Code')
    .select(
        'Billing_Document', 'Billing_Date', 'Year', 'Month', 'Sales_Office',
        'Model', 'Model_Name', 'Color', 'Type', 'VIN',
        'SlsVolQty', 'Sales_Price', 'Surcharge', 'Discount', 'Net_Sales',
        'Tax_Amount', 'Revenue_Re', 'Cost',
        'Dealer_Code', 'Dealer_Name', 'RM', 'ASE', 'Province', 'District',
        'Customer_Id', 'Age_At_Purchase',
    )
)

# Dealer commission (LKR per unit) - kept as a variable so it can be changed without touching the formula
Dealer_Commision = 15000

# Gross margin: Net_Sales - Tax_Amount - Revenue_Re - Cost - Dealer_Commision
df_gold_unit_sales = df_gold_unit_sales.withColumn(
    'Gross_Margin',
    F.round(
        F.col('Net_Sales') - F.col('Tax_Amount') - F.col('Revenue_Re') - F.col('Cost') - F.lit(Dealer_Commision),
        2
    )
)

# Place Gross_Margin right after Cost
cols = df_gold_unit_sales.columns
cols.remove('Gross_Margin')
i = cols.index('Cost') + 1
cols.insert(i, 'Gross_Margin')
df_gold_unit_sales = df_gold_unit_sales.select(*cols)

print(f"Rows: {df_gold_unit_sales.count():,} | Columns: {len(df_gold_unit_sales.columns)} | Dealer_Commision: {Dealer_Commision:,}")
df_gold_unit_sales.show(5, truncate=25)

df_gold_unit_sales.write.format("delta") \
    .mode("overwrite") \
    .option("mergeSchema", "true") \
    .saveAsTable(f"`{catalog_name}`.gold.gld_unit_sales")

+--------------------+--------------------+------------+--------------------+--------------------+--------------------+------------+----------------+----+----------+---------+----------+----------+-----------+-----------+------------+---------+----------+--------+---------+-----------+----------+----------+-----------+--------+----------+-----------+----------+-----------+------+----------+----------+-----+----------+----------+--------+----------+-----------+----------+------------------+---------+-----------------+----+-----+-----------+-----------+-------------+--------+-------+-------+--------+-----+--------------------+--------------------+----------+---------------+---------------+---------------+--------------------+--------------------+
|  Sales_Organization|Distribution_Channel|Sales_Office|      Sales_Employee|               Payer|            Material|Billing_Date|Billing_Document|Item|      Date|SlsVolQty|SlsVolQty2|Ad_Payment|Ad_Payment2|Sales_Price|Sales_Price2|Surcharge|S

In [0]:
# df = df_bronze_unit_sales.select("Material").distinct()

# model_list = {
#               "RAY ZR 125 STREET RALLY" , "Ray ZR Disc" , "RAY ZR 125 DISC" , "Ray ZR Street Rally",
#               "MT15 V2",
#               "FZS-FI V4(STD)" , "FZS-FI V4 (STD)" ,  "FZS-FI V4" , "FZ-S-FI NEW" , "FZS-DLX-149" , "FZ-S DLX 149",
#               "FZ FI V2",
#               "YZF-R15 V4", 
#               "FASINO S 125",
#               "XSR-155-DD10F" , "XSR-155-DD10G" , "XSR-155-DD10A",
#               "NMAX-10A-155"
# }

# import re

# # one pattern from model_list — longest names first so the most specific model wins
# pattern = "|".join(re.escape(m) for m in sorted(model_list, key=len, reverse=True))

# df_classified = (df
#     .withColumn("model", F.regexp_extract("Material", f"(?i)({pattern})", 1))
#     .withColumn("color", F.regexp_replace("Material", f"(?i){pattern}", ""))
#     # strip the symbols ( ) - . from the color values, then tidy up spacing
#     .withColumn("color", F.trim(F.regexp_replace(F.regexp_replace("color", "[()\\-.]", ""), "\\s+", " ")))
# )

# display(df_classified)

# model_anomalies = {
#     "FZS-FI V4(STD)" : "FZ-S FI V4 (STD)",
#     "FZS-FI V4 (STD)" : "FZ-S FI V4 (STD)",
#     "FZ-S-FI NEW" : "FZ-S FI NEW",
#     "FZS-DLX-149" : "FZ-S DLX",
#     "FZS-FI V4" : "FZ-S FI V4",
#     "RAY ZR 125 STREET RALLY" : "RAY ZR 125 Street Rally",
#     "RAY ZR 125 DISC" : "RAY ZR 125 Disc",
#     "MT15 V2" : "MT-15 V2",
#     "NMAX-10A-155" : "NMAX 155",
#     "XSR-155-DD10F" : "XSR 155",
#     "XSR-155-DD10G" : "XSR 155",
#     "XSR-155-DD10A" : "XSR 155"
# }

# # replace raw model names with their standardized names from model_anomalies
# df_classified = df_classified.replace(model_anomalies, subset=["model"])

# display(df_classified)

# color_anomalies = {
#     "DBNM8" : "DARK BLUISH GRAY METALLIC 8",
#     "DPBMY" : "DEEP PURPLISH BLUE METALLIC Y",
#     "MLNM4" : "MAT LIGHT GRAY METALLIC 4",
#     "MDNM9" : "MAT DARK GRAY METALLIC 9",
#     "MBL2" : "MAT BLACK 2",
#     "SM12" : "BLACK METALLIC 12",
#     "BMC" : "BLUE METALLIC C",
#     "MGM3" : "GRAYISH GREEN METALLIC 3",
#     "WMC" : "WHITE METALLIC C"
# }

# # replace raw color names with their standardized names from color_anomalies
# df_classified = df_classified.replace(color_anomalies, subset=["color"])

# display(df_classified)

### Dealers

In [0]:
df_silver_dealers = spark.table(f'`{catalog_name}`.silver.slv_dealers')
df_silver_dealers.show(10)

+-----------+--------------------+------------+---------+------------+---------------+------+---------+------------+--------------------+----+-----------------------+--------------------+--------------------+
|Dealer_Code|         Dealer_Name|    Province| District|         ASE|             RM| Admin|       BG|Credit_Limit|Cash_/_Credit_Status|Type|Status_Active/_Inactive|        _source_file|         ingested_at|
+-----------+--------------------+------------+---------+------------+---------------+------+---------+------------+--------------------+----+-----------------------+--------------------+--------------------+
|      10175|Cool Point Servic...|Sabaragamuwa|  Kegalle|Nisal Himasa|Jayantha - (SG)|Wasana| 700000.0|        NULL|              CREDIT|  MC|                 Active|dbfs:/Volumes/yam...|2026-11-01 09:11:...|
|      11346|Citizen Motors - ...|Sabaragamuwa|  Kegalle|Nisal Himasa|Jayantha - (SG)|Wasana| 400000.0|        NULL|              CREDIT|  MC|                 Activ

### Spareparts Orders

In [0]:
df_silver_spareparts_orders = spark.table(f'`{catalog_name}`.silver.slv_spareparts_orders')
df_silver_spareparts_orders.show(10)

# Simplest: blank strings → NULL, drop fully-null columns, drop rows missing essential fields
df_orders = df_silver_spareparts_orders.replace('', None)

n = df_orders.count()
nulls = df_orders.select([
    F.count(F.when(F.col(c).isNull(), c)).alias(c) for c in df_orders.columns
]).first().asDict()

df_spareparts_orders = (
    df_orders
    .drop(*[c for c, v in nulls.items() if v == n])              # fully-null columns
    .dropna(subset=[c for c, v in nulls.items() if 0 < v <= 15]) # rows missing essential fields
    .drop('Personnel_Number', 'Contract_Val_Period')             # constant columns (1 distinct value, 40% null)
)

print(f"Rows: {df_spareparts_orders.count():,} | Columns: {len(df_spareparts_orders.columns)}")

# Per column: distinct values, non-null cells, null/blank cells (blanks were already converted to NULL in Cell 5)
distinct_counts = df_spareparts_orders.select([
    F.countDistinct(F.col(c)).alias(c) for c in df_spareparts_orders.columns
]).first().asDict()

non_null = df_spareparts_orders.select([
    F.count(F.col(c)).alias(c) for c in df_spareparts_orders.columns
]).first().asDict()

n = df_spareparts_orders.count()

display(spark.createDataFrame(
    sorted(
        [(c, int(distinct_counts[c]), int(non_null[c]), n - int(non_null[c])) for c in df_spareparts_orders.columns],
        key=lambda t: t[1],
    ),
    ['column', 'distinct_values', 'value_cells', 'null_cells']
))

# Add Province, District, ASE, RM, Type from dealers to the CLEANED orders (left join on Sold-to_Party = Dealer_Code)
dealer_cols = ['Province', 'District', 'ASE', 'RM', 'Type']

joined = df_spareparts_orders.join(
    df_silver_dealers.select('Dealer_Code', *dealer_cols),
    on=df_spareparts_orders['Sold-to_Party'] == F.col('Dealer_Code'),
    how='left'
)

unmatched = joined.filter(F.col('Dealer_Code').isNull()).count()

# Place dealer columns right after AF_Open_Order_Quantity
cols = df_spareparts_orders.columns
i = cols.index('AF_Open_Order_Quantity') + 1
df_gold_spareparts_orders = joined.drop('Dealer_Code').select(*cols[:i], *dealer_cols, *cols[i:])

print(f"Rows: {df_gold_spareparts_orders.count():,} | Columns: {len(df_gold_spareparts_orders.columns)} (added {dealer_cols})")
print(f"Rows with no matching dealer: {unmatched:,}")

# Sales_Type: 'Returned' if Returns has a value; otherwise Credit/Cash from Sales_Document_Type_Description
df_gold_spareparts_orders = df_gold_spareparts_orders.withColumn(
    'Sales_Type',
    F.when(F.col('Returns').isNotNull(), 'Returned')
     .when(F.col('Sales_Document_Type_Description').contains('Credit'), 'Credit')
     .when(F.col('Sales_Document_Type_Description').contains('Cash'), 'Cash')
     .otherwise('Sale')
)

# Move Sales_Type next to Sales_Document_Type_Description
cols = df_gold_spareparts_orders.columns
cols.remove('Sales_Type')
i = cols.index('Sales_Document_Type_Description') + 1
df_gold_spareparts_orders = df_gold_spareparts_orders.select(*cols[:i], 'Sales_Type', *cols[i:])

df_gold_spareparts_orders.groupBy('Sales_Type').count().show()

# Restructure columns: keep only the key business columns (front_cols), drop the rest
front_cols = [
    'Sales_Document', 'Material', 'Material_Description',
    'Order_Quantity_Item', 'Confirmed_Quantity_Item',
    'Net_Price', 'Net_Value_Item', 'Net_Value_Header', 'Sales_Type',
    'Sold-to_Party', 'Sold-To_Party_Name', 'Type', 'RM', 'ASE', 'Province', 'District',
    'Returns', 'Order_Reason', 'Order_Reason_Description',
    'Reason_for_Rejection', 'Rejection_Reason_Description',
    'Document_Date','Delivery_Date', 'Goods_Issue_Date', 'Material_Availability_Date',
]

missing = [c for c in front_cols if c not in df_gold_spareparts_orders.columns]
assert not missing, f"Columns not found: {missing}"

# Keep only the front columns - everything else is dropped
df_gold_spareparts_orders = df_gold_spareparts_orders.select(*front_cols)

# Lost quantity and lost value per order item
# Confirmed_Quantity_Item is a string with decimals ('0.3') and comma thousands ('1,500'), so strip commas then cast to double;
# Lost_Quantity_Item rounds to an integer
df_gold_spareparts_orders = (
    df_gold_spareparts_orders
    .withColumn('Lost_Quantity_Item', F.round(F.col('Order_Quantity_Item') - F.regexp_replace(F.col('Confirmed_Quantity_Item'), ',', '').cast('double')).cast('int'))
    .withColumn('Lost_Value_Item', F.round(F.col('Lost_Quantity_Item') * F.col('Net_Price'), 2))
)

# Place Lost_Quantity_Item after Confirmed_Quantity_Item, Lost_Value_Item after Net_Value_Header
cols = df_gold_spareparts_orders.columns
cols.remove('Lost_Quantity_Item')
cols.remove('Lost_Value_Item')
i = cols.index('Confirmed_Quantity_Item') + 1
cols.insert(i, 'Lost_Quantity_Item')
j = cols.index('Net_Value_Header') + 1
cols.insert(j, 'Lost_Value_Item')
df_gold_spareparts_orders = df_gold_spareparts_orders.select(*cols)

df_gold_spareparts_orders.show(10)

# Split gold orders by dealer Type: MC, OBM, and everything else (null/unknown)
df_gold_spareparts_orders_mc = df_gold_spareparts_orders.filter(F.col('Type') == 'MC')
df_gold_obm_spareparts_orders = df_gold_spareparts_orders.filter(F.col('Type') == 'OBM')
df_gold_other_spareparts_orders = df_gold_spareparts_orders.filter(F.col('Type').isNull())

print(f"MC:    {df_gold_spareparts_orders_mc.count():,} rows")
print(f"OBM:   {df_gold_obm_spareparts_orders.count():,} rows")
print(f"Other: {df_gold_other_spareparts_orders.count():,} rows (null Type = unmatched dealers)")

df_gold_spareparts_orders.groupBy('Type').count().show()

# Derive Material_Type from Material_Description (MC orders only)
df_gold_spareparts_orders_mc = df_gold_spareparts_orders_mc.withColumn(
    'Material_Type',
    F.when(F.col('Material_Description').contains('YAMALUBE'), 'Lubricant')
     .when(F.col('Material_Description').contains('KARATE BATTERY'), 'Battery')
     .when(F.col('Material_Description').contains('KATANA TYRE'), 'Tyre')
     .otherwise('MC Spareparts')
)

# Move Material_Type next to Material_Description
cols = df_gold_spareparts_orders_mc.columns
cols.remove('Material_Type')
i = cols.index('Material_Description') + 1
df_gold_spareparts_orders_mc = df_gold_spareparts_orders_mc.select(*cols[:i], 'Material_Type', *cols[i:])

df_gold_spareparts_orders_mc.groupBy('Material_Type').count().show()

# Split MC orders by Material_Type into one DataFrame per type
df_gold_tyre_orders = df_gold_spareparts_orders_mc.filter(F.col('Material_Type') == 'Tyre')
df_gold_lubricant_orders = df_gold_spareparts_orders_mc.filter(F.col('Material_Type') == 'Lubricant')
df_gold_battery_orders = df_gold_spareparts_orders_mc.filter(F.col('Material_Type') == 'Battery')
df_gold_mc_spareparts_orders = df_gold_spareparts_orders_mc.filter(F.col('Material_Type') == 'MC Spareparts')

print(f"Tyre:         {df_gold_tyre_orders.count():,} rows")
print(f"Lubricant:    {df_gold_lubricant_orders.count():,} rows")
print(f"Battery:      {df_gold_battery_orders.count():,} rows")
print(f"MC Spareparts: {df_gold_mc_spareparts_orders.count():,} rows")

df_gold_tyre_orders.show(10)
df_gold_lubricant_orders.show(10)
df_gold_battery_orders.show(10)
df_gold_mc_spareparts_orders.show(10)

df_gold_tyre_orders.write.format("delta") \
    .mode("overwrite") \
    .option("mergeSchema", "true") \
    .saveAsTable(f"`{catalog_name}`.gold.gld_tyre_orders")

df_gold_lubricant_orders.write.format("delta") \
    .mode("overwrite") \
    .option("mergeSchema", "true") \
    .saveAsTable(f"`{catalog_name}`.gold.gld_lubricant_orders")

df_gold_battery_orders.write.format("delta") \
    .mode("overwrite") \
    .option("mergeSchema", "true") \
    .saveAsTable(f"`{catalog_name}`.gold.gld_battery_orders")

df_gold_mc_spareparts_orders.write.format("delta") \
    .mode("overwrite") \
    .option("mergeSchema", "true") \
    .saveAsTable(f"`{catalog_name}`.gold.gld_mc_spareparts_orders")

df_gold_obm_spareparts_orders.write.format("delta") \
    .mode("overwrite") \
    .option("mergeSchema", "true") \
    .saveAsTable(f"`{catalog_name}`.gold.gld_obm_spareparts_orders")

+------------------+-------------+-------------------+--------------+-------------------+-------------+---------------+-------------------+----------+--------------+-----------------+------------+-------------------------+----------+----------+--------+-------------+--------------------------+------------------+--------------+----------------+--------+--------------------+------------+-----------+------------------+--------------------+--------------+---------------------------------+--------------------+--------------------+--------------------+-----+-----------------------+------------+----------------+----------------+--------------------+---------+-------+---------------------------+-----+---------------+-------------------+----------------------------+-------------+------------+--------------------------+----------------+----------------+--------------+-------+--------------------+-------------+----------------------------+--------------------------------+-------------------------

column,distinct_values,value_cells,null_cells
Document_Currency,1,155405,0
Sales_Office,1,155405,0
Sales_Organization,1,155405,0
Distribution_Channel,1,155405,0
Pricing_Unit_per,1,155405,0
Returns,1,1949,153456
Target_Quantity,1,155405,0
Sales_Organization_Name,1,155405,0
Distribution_Channel_Name,1,155405,0
_source_file,1,155405,0


Rows: 155,406 | Columns: 72 (added ['Province', 'District', 'ASE', 'RM', 'Type'])
Rows with no matching dealer: 9,669
+----------+-----+
|Sales_Type|count|
+----------+-----+
|    Credit|85738|
|  Returned| 1950|
|      Cash|67718|
+----------+-----+

+--------------+---------------+--------------------+-------------------+-----------------------+------------------+---------+--------------+----------------+---------------+----------+-------------+--------------------+----+-----------------+-------------------+--------+--------+-------+------------+------------------------+--------------------+----------------------------+-------------+-------------+----------------+--------------------------+
|Sales_Document|       Material|Material_Description|Order_Quantity_Item|Confirmed_Quantity_Item|Lost_Quantity_Item|Net_Price|Net_Value_Item|Net_Value_Header|Lost_Value_Item|Sales_Type|Sold-to_Party|  Sold-To_Party_Name|Type|               RM|                ASE|Province|District|Returns|Order_Rea

### Spareparts Sales

In [0]:
df_silver_spareparts_sales = spark.table(f'`{catalog_name}`.silver.slv_spareparts_sales')
df_silver_spareparts_sales.show(10)

# Split Payer at the first space into Dealer_Id and Dealer_Name (trim the padding spaces)
parts = F.split(F.col('Payer'), ' ', 2)
df_silver_spareparts_sales = (
    df_silver_spareparts_sales
    .withColumn('Dealer_Id', parts[0])
    .withColumn('Dealer_Name', F.trim(F.try_element_at(parts, F.lit(2))))
)

# Place Dealer_Id and Dealer_Name right after Payer
cols = df_silver_spareparts_sales.columns
cols.remove('Dealer_Id')
cols.remove('Dealer_Name')
i = cols.index('Payer') + 1
cols[i:i] = ['Dealer_Id', 'Dealer_Name']
df_silver_spareparts_sales = df_silver_spareparts_sales.select(*cols)

df_silver_spareparts_sales.select('Payer', 'Dealer_Id', 'Dealer_Name').show(10, truncate=False)

# Split Material at the first space: code stays in Material, description goes to Material_Description (trim padding)
parts = F.split(F.col('Material'), ' ', 2)
df_silver_spareparts_sales = (
    df_silver_spareparts_sales
    .withColumn('Material_Description', F.trim(F.try_element_at(parts, F.lit(2))))
    .withColumn('Material', parts[0])
)

# Place Material_Description right after Material
cols = df_silver_spareparts_sales.columns
cols.remove('Material_Description')
i = cols.index('Material') + 1
cols.insert(i, 'Material_Description')
df_silver_spareparts_sales = df_silver_spareparts_sales.select(*cols)

# Drop the original Payer column (replaced by Dealer_Id and Dealer_Name)
df_silver_spareparts_sales = df_silver_spareparts_sales.drop('Payer')

df_silver_spareparts_sales.select('Material', 'Material_Description').show(10, truncate=False)

# Restructure columns: keep only the key business columns (front_cols), drop the rest
front_cols = [
    'Billing_Document', 'Billing_Date', 'Material', 'Material_Description',
    'SlsVolQty', 'Sales_Pric', 'Surcharge', 'Discount', 'Net_Sales', 'Tax_Amount' , 'Cost',
    'Dealer_Id', 'Dealer_Name',
]

missing = [c for c in front_cols if c not in df_silver_spareparts_sales.columns]
assert not missing, f"Columns not found: {missing}"

# Keep only the front columns - everything else is dropped
df_silver_spareparts_sales = df_silver_spareparts_sales.select(*front_cols)

df_silver_spareparts_sales.show(5)

# Billing calendar columns derived from Billing_Date
df_silver_spareparts_sales = (
    df_silver_spareparts_sales
    .withColumn('Billing_Year', F.year('Billing_Date'))
    .withColumn('Billing_Month', F.month('Billing_Date'))
    .withColumn('Billing_Week', F.weekofyear('Billing_Date'))
)

# Place Billing_Year, Billing_Month, Billing_Week right after Billing_Date
cols = df_silver_spareparts_sales.columns
cols.remove('Billing_Year')
cols.remove('Billing_Month')
cols.remove('Billing_Week')
i = cols.index('Billing_Date') + 1
cols[i:i] = ['Billing_Year', 'Billing_Month', 'Billing_Week']
df_silver_spareparts_sales = df_silver_spareparts_sales.select(*cols)

# Add Province, District, ASE, RM, Type from dealers to the sales (left join on Dealer_Id = Dealer_Code)
dealer_cols = ['Type', 'RM', 'ASE', 'Province', 'District']

# Drop any pre-existing dealer columns so this cell can be re-run without creating duplicates
df_silver_spareparts_sales = df_silver_spareparts_sales.drop(*dealer_cols)

before = df_silver_spareparts_sales.count()

joined = df_silver_spareparts_sales.join(
    df_silver_dealers.select('Dealer_Code', *dealer_cols),
    on=df_silver_spareparts_sales['Dealer_Id'] == F.col('Dealer_Code'),
    how='left'
)

unmatched = joined.filter(F.col('Dealer_Code').isNull()).count()

# Place dealer columns right after Dealer_Name
cols = df_silver_spareparts_sales.columns
i = cols.index('Dealer_Name') + 1
df_silver_spareparts_sales = joined.drop('Dealer_Code').select(*cols[:i], *dealer_cols, *cols[i:])

after = df_silver_spareparts_sales.count()
print(f"Rows: {after:,} | Columns: {len(df_silver_spareparts_sales.columns)} (added {dealer_cols})")
print(f"Rows with no matching dealer: {unmatched:,}")
if after != before:
    print(f"WARNING: row count changed {before:,} -> {after:,} (fan-out from duplicate Dealer_Code)")

df_silver_spareparts_sales.show(10)

# Split spareparts sales by dealer Type: MC, OBM, and everything else (includes null = unmatched dealers)
df_gold_spareparts_sales_mc = df_silver_spareparts_sales.filter(F.col('Type') == 'MC')
df_gold_obm_spareparts_sales = df_silver_spareparts_sales.filter(F.col('Type') == 'OBM')
df_gold_other_spareparts_sales = df_silver_spareparts_sales.filter(F.col('Type').isNull() | ~F.col('Type').isin('MC', 'OBM'))

print(f"MC:    {df_gold_spareparts_sales_mc.count():,} rows")
print(f"OBM:   {df_gold_obm_spareparts_sales.count():,} rows")
print(f"Other: {df_gold_other_spareparts_sales.count():,} rows (includes null Type = unmatched dealers)")

df_silver_spareparts_sales.groupBy('Type').count().show()

df_gold_spareparts_sales_mc.show(5)
df_gold_obm_spareparts_sales.show(5)
df_gold_other_spareparts_sales.show(5)

# Derive Material_Type from Material_Description (MC sales only)
df_gold_spareparts_sales_mc = df_gold_spareparts_sales_mc.withColumn(
    'Material_Type',
    F.when(F.col('Material_Description').contains('YAMALUBE'), 'Lubricant')
     .when(F.col('Material_Description').contains('KARATE BATTERY'), 'Battery')
     .when(F.col('Material_Description').contains('KATANA TYRE'), 'Tyre')
     .otherwise('MC Spareparts')
)

# Move Material_Type next to Material_Description
cols = df_gold_spareparts_sales_mc.columns
cols.remove('Material_Type')
i = cols.index('Material_Description') + 1
cols.insert(i, 'Material_Type')
df_gold_spareparts_sales_mc = df_gold_spareparts_sales_mc.select(*cols)

df_gold_spareparts_sales_mc.groupBy('Material_Type').count().show()

# Split MC sales by Material_Type into one DataFrame per type
df_gold_tyre_sales = df_gold_spareparts_sales_mc.filter(F.col('Material_Type') == 'Tyre')
df_gold_lubricant_sales = df_gold_spareparts_sales_mc.filter(F.col('Material_Type') == 'Lubricant')
df_gold_battery_sales = df_gold_spareparts_sales_mc.filter(F.col('Material_Type') == 'Battery')
df_gold_mc_spareparts_sales = df_gold_spareparts_sales_mc.filter(F.col('Material_Type') == 'MC Spareparts')

print(f"Tyre:         {df_gold_tyre_sales.count():,} rows")
print(f"Lubricant:    {df_gold_lubricant_sales.count():,} rows")
print(f"Battery:      {df_gold_battery_sales.count():,} rows")
print(f"MC Spareparts: {df_gold_mc_spareparts_sales.count():,} rows")

df_gold_tyre_sales.show(5)
df_gold_lubricant_sales.show(5)
df_gold_battery_sales.show(5)
df_gold_mc_spareparts_sales.show(5)

df_gold_tyre_sales.write.format("delta") \
    .mode("overwrite") \
    .option("mergeSchema", "true") \
    .saveAsTable(f"`{catalog_name}`.gold.gld_tyre_sales")

df_gold_lubricant_sales.write.format("delta") \
    .mode("overwrite") \
    .option("mergeSchema", "true") \
    .saveAsTable(f"`{catalog_name}`.gold.gld_lubricant_sales")

df_gold_battery_sales.write.format("delta") \
    .mode("overwrite") \
    .option("mergeSchema", "true") \
    .saveAsTable(f"`{catalog_name}`.gold.gld_battery_sales")

df_gold_mc_spareparts_sales.write.format("delta") \
    .mode("overwrite") \
    .option("mergeSchema", "true") \
    .saveAsTable(f"`{catalog_name}`.gold.gld_mc_spareparts_sales")

df_gold_obm_spareparts_sales.write.format("delta") \
    .mode("overwrite") \
    .option("mergeSchema", "true") \
    .saveAsTable(f"`{catalog_name}`.gold.gld_obm_spareparts_sales")

+--------------------+--------------------+------------------+--------------------+--------------------+--------------------+------------+----------------+------+----------+---------+----------+----------+-----------+----------+-----------+---------+----------+----------+---------+----------+----------+----------+-----------+--------+----------+-----------+----------+-----------+-----+----------+---------+-----+----------+----------+--------+----------+-----------+-----------+----------+---------+--------------------+--------------------+
|  Sales_Organization|Distribution_Channel|      Sales_Office|      Sales_Employee|               Payer|            Material|Billing_Date|Billing_Document|  Item|      Date|SlsVolQty|SlsVolQty2|Ad_Payment|Ad_Payment2|Sales_Pric|Sales_Pric2|Surcharge|Surcharge2|  Discount|Discount2| Net_Sales|Net_Sales2|Tax_Amount|Tax_Amount2|Tax_Code|Revenue_Re|Revenue_Re2|Cost_Reduc|Cost_Reduc2|Batch|Bill__Type|     Cost|Cost2|Cust__Clas|CustGrpOrd|Division|Matl_Grou

### Stock Status

In [0]:
df_silver_supersession = spark.table(f'`{catalog_name}`.silver.slv_supersession')
df_silver_supersession.show(10)

df_silver_current_stock = spark.table(f'`{catalog_name}`.silver.slv_current_stock')
df_silver_current_stock.show(10)

df_silver_on_orders = spark.table(f'`{catalog_name}`.silver.slv_on_orders')
df_silver_on_orders.show(10)

# Stock status: match the current stock Material against BOTH supersession keys - Latest_SS (priority)
# and the original Material (fallback). If the Latest_SS part is stocked anywhere, its stock is used;
# otherwise the original Material's stock is used.
stock_cols = ['Name_1', 'Plant', 'Storage_Location', 'Descr__of_Storage_Loc_', 'Unrestricted']
added_cols = [*stock_cols, *[f'Orig_{c}' for c in stock_cols]]

# Defensive drop so this cell can be re-run without creating duplicate columns
df_silver_supersession = df_silver_supersession.drop(*added_cols)

stock = df_silver_current_stock.select(F.col('Material').alias('Stock_Key'), *stock_cols)
stocked_keys = df_silver_current_stock.select(F.col('Material').alias('Stocked_Key')).distinct()

# Effective key: Latest_SS when that part is stocked, otherwise the original Material
df_silver_supersession = (
    df_silver_supersession
    .join(stocked_keys, df_silver_supersession['Latest_SS'] == F.col('Stocked_Key'), 'left')
    .withColumn(
        'Effective_Key',
        F.when(F.col('Stocked_Key').isNotNull(), F.col('Latest_SS')).otherwise(F.col('Material'))
    )
    .drop('Stocked_Key')
)

via_latest = df_silver_supersession.filter(F.col('Effective_Key') == F.col('Latest_SS')).count()
via_material = df_silver_supersession.filter(F.col('Effective_Key') != F.col('Latest_SS')).count()

before = df_silver_supersession.count()

df_silver_supersession = (
    df_silver_supersession
    .join(stock, F.col('Effective_Key') == F.col('Stock_Key'), 'left')
    .drop('Stock_Key', 'Effective_Key')
)

after = df_silver_supersession.count()
print(f"Rows: {before:,} -> {after:,} | Columns: {len(df_silver_supersession.columns)} (added stock matched by Latest_SS first, then original Material)")
print(f"Materials matched via Latest_SS: {via_latest:,}")
print(f"Materials matched via Material:  {via_material:,} (fallback - Latest_SS not stocked)")
print(f"Rows with no stock at all:      {df_silver_supersession.filter(F.col('Name_1').isNull()).count():,}")

if after != before:
    print(f"WARNING: row count changed {before:,} -> {after:,} (fan-out from multiple stock rows per Material)")

df_silver_supersession.show(10, truncate=25)

# On orders: bring the monthly on-order quantities of the Latest_SS part into the stock-status frame
# (left join on Latest_SS = on-orders Material, consistent with the stock joins in the previous cells)
order_cols = ['Sep_2026', 'Oct_2026', 'Nov_2026', 'Dec_2026']

# Defensive drop so this cell can be re-run without creating duplicate columns
df_silver_supersession = df_silver_supersession.drop(*order_cols)

on_orders = df_silver_on_orders.select(
    F.col('Material').alias('SS_Latest'), *order_cols
)

before = df_silver_supersession.count()

df_silver_supersession = (
    df_silver_supersession
    .join(on_orders, df_silver_supersession['Latest_SS'] == F.col('SS_Latest'), 'left')
    .drop('SS_Latest')
)

after = df_silver_supersession.count()
print(f"Rows: {before:,} -> {after:,} | Columns: {len(df_silver_supersession.columns)} (added on-order quantities for Latest_SS)")
print(f"Rows with no on-order record for the Latest_SS part: {df_silver_supersession.filter(F.col('Sep_2026').isNull()).count():,}")

if after != before:
    print(f"WARNING: row count changed {before:,} -> {after:,} (fan-out from multiple on-order rows per Material)")

df_silver_stock_position = df_silver_supersession.drop('_source_file', 'ingested_at')

df_silver_stock_position.show(10)

df_silver_stock_position.write.format("delta") \
    .mode("overwrite") \
    .option("mergeSchema", "true") \
    .saveAsTable(f"`{catalog_name}`.gold.gld_stock_position")

+---------------+---------------+--------------------+-------------+--------------+-----+-------------+-------------+-------------+-------------+-------------+-------------+-------------+-------------+-------------+--------------+--------------------+--------------------+
|       Material|      Latest_SS|Material_description|Material_Type|Material_Group|Brand|1st_Supersede|2nd_Supersede|3rd_Supersede|4th_Supersede|5th_Supersede|6th_Supersede|7th_Supersede|8th_Supersede|9th_Supersede|10th_Supersede|        _source_file|         ingested_at|
+---------------+---------------+--------------------+-------------+--------------+-----+-------------+-------------+-------------+-------------+-------------+-------------+-------------+-------------+-------------+--------------+--------------------+--------------------+
|  0NH9-28471-00|  0NH9-28471-00|   BRKT.PATROL LIGHT|         ERSA|     AWPYM0027|   YM|         NULL|         NULL|         NULL|         NULL|         NULL|         NULL|        